# DataFrames en Spark

Iniciar Sesión de Spark (Spark Session)

---

In [ ]:
from pyspark.sql import SparkSession

# crear la sesión
spark = SparkSession \
        .builder \
        .appName("DataFrames Basics") \
        .master("local[*]") \
        .getOrCreate()

spark.version

In [ ]:
spark

In [ ]:
# Para optimización de conversión a Pandas
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [ ]:
# Importar funciones sql
from pyspark.sql.functions import *

## Ejemplos

In [ ]:
from pyspark.sql.types import Row
from pyspark.sql.functions import *

Crear una RDD directamente a través de un CSV

In [ ]:
bankText = spark.sparkContext.textFile("bank.csv")

#Tenemos que eliminar la primera fila porque es la de los encabezados
bank = bankText.map(lambda lineaCsv: lineaCsv.split(";"))\
.filter(lambda s: s[0] != "\"age\"") \
.map(lambda row: Row(int(row[0]), row[1].replace("\"", ""), row[2].replace("\"", ""), row[3].replace("\"", ""), row[5].replace("\"", ""))) \
.toDF(["age", "job", "marital", "education", "balance"]) \
.withColumn("age", col("age").cast("int"))

bank.show(3)

Leer directamente desde un archivo JSON a un DF

In [ ]:
guitarPlayersDF = spark.read.option("inferSchema", True).json("guitarPlayers.json") # inferSchema requiere ponerlo a True para ser usado
# si no se establece None, utiliza el valor por defecto (default = False) también puedes pasar el esquema manualmente

Leer directamente desde csv

In [ ]:
booksDF = spark.read.option("header", "true") \
                   .option("delimiter", ",") \
                   .csv("books.csv")

booksDF.show(3)


Mostrar un DF e imprimir el esquema

In [ ]:
guitarPlayersDF.show(2)
guitarPlayersDF.printSchema()

Obtener Filas

In [ ]:
guitarPlayersDF.take(2)

Conteo

In [ ]:
guitarPlayersDF.count()

Esquema

In [ ]:
# obtener un esquema
guitarPlayersSchema = guitarPlayersDF.schema
print(type(guitarPlayersSchema))
print(guitarPlayersSchema)

Esquemas Personalizados

In [ ]:
players_rdd = spark.sparkContext.parallelize([
    ("Leo Messi", "Delantero", 10),
    ("Virgil van Dijk", "Defensa", 4),
    ("David Villa", "Delantero", 7)
])

In [ ]:
exampleDF = spark.createDataFrame(players_rdd)
exampleDF.printSchema()

Con nombres de columna

In [ ]:
names = list(["Nombre", "Posicion", "Numero"])

In [ ]:
example2DF = players_rdd.toDF(names)
example2DF.printSchema()

In [ ]:
# importar tipos sql
from pyspark.sql.types import *

In [ ]:
# esquema personalizado
customSchema = StructType([ \
    StructField('Nombre', StringType(), True), \
    StructField('Posicion', StringType(), True), \
    StructField('Numero', StringType(), True)])

In [ ]:
example3DF = spark.createDataFrame(players_rdd, customSchema)
example3DF.printSchema()

In [ ]:
example3DF.show(2, False)

In [ ]:
# también podemos especificar el esquema con DDL (Data Definition Language)
customSchema2 = "`Nombre` STRING NOT NULL, `Posicion` STRING, `Numero` INT"

In [ ]:
example4DF = spark.createDataFrame(players_rdd, customSchema2)
example4DF.printSchema()

In [ ]:
print(type(example2DF.collect()[0]["Numero"]))
print(type(example3DF.collect()[0]["Numero"]))

## Ejercicios

1. Crea un DF manual describiendo bebidas (beverages) 🍹

    fabricante
    sabor
    tipo_de_envase
    contenido_de_azúcar_gramos

2. Carga cualquier otro archivo de la carpeta de datos 🏙️

    imprime su esquema
    cuenta el número de filas, llama a `count()`

3. Echa un vistazo a taxi_zones.csv. Lee el archivo a un DF, pero esta vez con tu propio esquema 🎤

Ejercicio 1

Ejercicio 2

Ejercicio 3